# Backtrader: realistic limit fills in one line

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sarptandoven/queuetruth/blob/main/examples/notebooks/07_backtrader.ipynb)

Backtrader fills a limit order as soon as a bar reaches its price. In the real queue many of those orders would not
have filled. queuetruth's `QueueFiller` keeps Backtrader's own limit logic and fills each touched order with the
probability the atlas touch tables give for its tier, side and bar length; `OrderLog` writes the orders so
`qt.check` can say how wrong the run was:

```python
cerebro.broker.set_filler(qt.backtrader.QueueFiller(tier="T1"))
cerebro.addanalyzer(qt.backtrader.OrderLog)
```

Data: queuetruth's **synthetic** dataset and the repository's **synthetic** atlas (invented numbers). The numbers
only show the workflow; they say nothing about a real venue.

In [1]:
# Setup: on Colab (or anywhere queuetruth is missing) this installs it from PyPI; elsewhere it does nothing.
import importlib
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("queuetruth") is None:
    try:
        from IPython import get_ipython

        shell = get_ipython()
    except ImportError:
        shell = None
    if shell is not None:  # %pip lets Colab offer a restart when an already-imported package was upgraded
        shell.run_line_magic("pip", "install -q queuetruth pandas matplotlib")
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "queuetruth", "pandas", "matplotlib"])
    importlib.invalidate_caches()
if importlib.util.find_spec("queuetruth") is None or importlib.util.find_spec("queuetruth.frames") is None:
    raise RuntimeError(
        "this notebook needs queuetruth.frames (first released after queuetruth 0.1.1): pip install -U queuetruth"
    )

In [2]:
import importlib.util
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

import queuetruth as qt

pd.set_option("display.width", 140)
WORK = Path("qt-notebooks") / "07_backtrader"
WORK.mkdir(parents=True, exist_ok=True)

# queuetruth's synthetic order-level dataset: two futures (QTAU6, QTBU6), two days, invented prices.
DATA = WORK / "synth"
if not (DATA / "manifest.json").exists():
    qt.fixtures_synth(str(DATA))

# The repository's synthetic atlas (invented numbers, no market data): the checkout's copy, else the one on GitHub.
# Leave `source=` out to use the published atlas (it asks before downloading anything).
_local = Path(qt.__file__).resolve().parents[2] / "tests" / "fixtures" / "atlas_synth"
ATLAS = (
    str(_local)
    if (_local / "atlas" / "v1" / "manifest.json").exists()
    else "https://raw.githubusercontent.com/sarptandoven/queuetruth/main/tests/fixtures/atlas_synth/atlas/v1/"
)

# Twelve resting limit orders on the synthetic futures, as a bar backtest reports them: placed at `placed` (UTC),
# and filled 5 s later when `filled` (the touch rule), else cancelled 30 s later.
ORDERS = pd.DataFrame(
    [
        ("O-1", "QTBU6", "sell", 1, 5008.50, "2026-06-16 14:38:42", True),
        ("O-2", "QTAU6", "buy", 2, 4999.75, "2026-06-16 14:38:05", False),
        ("O-3", "QTBU6", "sell", 3, 5010.25, "2026-06-15 17:09:27", True),
        ("O-4", "QTAU6", "buy", 1, 4999.50, "2026-06-16 17:32:15", False),
        ("O-5", "QTAU6", "sell", 2, 5002.75, "2026-06-15 15:39:13", True),
        ("O-6", "QTAU6", "buy", 3, 5004.00, "2026-06-15 19:04:05", False),
        ("O-7", "QTBU6", "buy", 1, 5008.75, "2026-06-15 15:36:19", True),
        ("O-8", "QTBU6", "sell", 2, 5008.50, "2026-06-16 16:58:59", False),
        ("O-9", "QTAU6", "buy", 3, 4999.50, "2026-06-16 14:39:20", True),
        ("O-10", "QTAU6", "buy", 1, 4999.50, "2026-06-16 17:25:05", False),
        ("O-11", "QTBU6", "buy", 2, 5008.25, "2026-06-16 16:12:29", True),
        ("O-12", "QTAU6", "buy", 3, 5001.25, "2026-06-15 15:05:05", False),
    ],
    columns=["order", "symbol", "side", "qty", "price", "placed", "filled"],
)
ORDERS["placed"] = pd.to_datetime(ORDERS["placed"], utc=True)
ORDERS["done"] = ORDERS["placed"] + pd.to_timedelta(ORDERS["filled"].map({True: 5, False: 30}), unit="s")


def have(*modules, hint=None):
    """True when every module is importable; else prints which are missing and how to get them."""
    missing = [m for m in modules if importlib.util.find_spec(m) is None]
    if missing:
        print(f"skipped: {', '.join(missing)} not installed here ({hint or 'pip install ' + ' '.join(missing)})")
    return not missing


print("queuetruth", qt.__version__, "| SYNTHETIC data and a SYNTHETIC atlas: the numbers show the workflow only")

queuetruth 0.3.0 | SYNTHETIC data and a SYNTHETIC atlas: the numbers show the workflow only


## 1. What the filler draws from

The touch tables give `P(real | touch)`: the probability that a resting limit order the bar touched really filled
in the queue, by tier, side and bar length. The filler is made from the atlas's tables (here the synthetic atlas;
without `atlas=` the published one, after asking) and needs no Backtrader to inspect.

In [3]:
filler = qt.backtrader.QueueFiller(tier="T1", atlas=ATLAS, yes=True)
display(
    pd.DataFrame(
        [
            {"bar": b, "side": s, "P(real | touch)": filler.touch.probability(b, s)}
            for b in filler.touch.bars
            for s in ("buy", "sell")
        ]
    )
)

,bar,side,P(real | touch)
0,1h,buy,0.65
1,1h,sell,0.67
2,1m,buy,0.35
3,1m,sell,0.37
4,5m,buy,0.50
5,5m,sell,0.52


## 2. The one line

A small strategy on synthetic 5-minute bars quotes one tick under the close every hour. With Backtrader installed
the cell runs it with the filler and the order log; without it, it says so.

In [4]:
if have("backtrader"):
    import backtrader as bt
    import numpy as np

    idx = pd.date_range("2026-06-15 14:00", periods=288, freq="5min")
    close = 5000 + np.cumsum(np.random.default_rng(0).normal(0, 0.5, len(idx)))
    bars = pd.DataFrame(
        {"open": close, "high": close + 0.5, "low": close - 0.5, "close": close, "volume": 100}, index=idx
    )

    class QuoteUnderTheClose(bt.Strategy):
        def next(self):
            if len(self) % 12 == 0:
                self.buy(exectype=bt.Order.Limit, price=self.data.close[0] - 0.25, size=1)

    cerebro = bt.Cerebro()
    cerebro.adddata(bt.feeds.PandasData(dataname=bars, timeframe=bt.TimeFrame.Minutes, compression=5), name="QTAU6")
    cerebro.addstrategy(QuoteUnderTheClose)
    run_filler = qt.backtrader.QueueFiller(tier="T1", atlas=ATLAS, yes=True)
    cerebro.broker.set_filler(run_filler)
    cerebro.addanalyzer(qt.backtrader.OrderLog, path=str(WORK / "bt_run.jsonl"))
    cerebro.run()
    print(run_filler.summary())

skipped: backtrader not installed here (pip install backtrader)


## 3. How wrong was the run?

`qt.check` reads the order log. Here it is twelve orders on the synthetic futures, written as `OrderLog` writes
them (one line per notification); use your own log. Without order-level data the answer is the atlas estimate:

In [5]:
log = WORK / "bt_orders.jsonl"


def stamp(t):
    return t.strftime("%Y-%m-%dT%H:%M:%S")


with log.open("w") as f:
    for k, o in enumerate(ORDERS.itertuples(), start=1):
        size = o.qty if o.side == "buy" else -o.qty
        base = {"event": "order", "ref": k, "data": o.symbol, "ordtype": o.side.title(), "exectype": "Limit",
                "size": size, "price": o.price, "pricelimit": o.price, "created": stamp(o.placed), "valid": None,
                "order_key": None}  # fmt: skip
        lines = [{**base, "status": s, "dt": stamp(o.placed), "exbits": []} for s in ("Submitted", "Accepted")]
        if o.filled:
            bit = {"dt": stamp(o.done), "size": size, "price": o.price, "comm": 0}
            lines.append({**base, "status": "Completed", "dt": stamp(o.done), "exbits": [bit]})
        else:
            lines.append({**base, "status": "Canceled", "dt": stamp(o.done), "exbits": []})
        f.writelines(json.dumps(x) + "\n" for x in lines)

estimate = qt.check(log, bar="5m", tier="T1", atlas=ATLAS, yes=True, out=WORK / "estimate")
display(estimate)

tier,tier_name,side,bar,orders,backtest_filled,p_real_given_touch,p_real_given_touch_ci_lo,p_real_given_touch_ci_hi,expected_real_fills,pnl_overstated_ticks_per_order,pnl_overstated_ticks_per_order_ci_lo,pnl_overstated_ticks_per_order_ci_hi,atlas_orders,atlas_dates,atlas_symbols
T1_top100,the 100 most traded symbols,buy,5m,8,3,0.5,0.45,0.55,1.5,0.4,0.3,0.5,4000,3,40
T1_top100,the 100 most traded symbols,sell,5m,4,3,0.52,0.47,0.57,1.56,0.4,0.3,0.5,4000,3,40


With order-level data of the same days, each order is replayed in the exact queue instead:

In [6]:
res = qt.check(log, data=DATA, horizon="1s", window="10s", out=WORK / "check")
display(res)

order,symbol,side,price,qty,entry,entry_local,backtest_filled,backtest_fill_after_s,real_filled,real_fill_after_s,queue_ahead,outcome,pnl_gap
sim:backtrader:backtest:1,QTBU6,sell,5008.5,1,2026-06-16T14:38:42.000+00:00,2026-06-16 14:38:42.000 UTC,True,5,True,2.497,1,both,0
sim:backtrader:backtest:10,QTAU6,buy,4999.5,1,2026-06-16T17:25:05.000+00:00,2026-06-16 17:25:05.000 UTC,False,,False,,12,neither,0
sim:backtrader:backtest:11,QTBU6,buy,5008.25,2,2026-06-16T16:12:29.000+00:00,2026-06-16 16:12:29.000 UTC,True,5,False,,0,phantom,25
sim:backtrader:backtest:12,QTAU6,buy,5001.25,3,2026-06-15T15:05:05.000+00:00,2026-06-15 15:05:05.000 UTC,False,,True,0.901,17,real_only,0
sim:backtrader:backtest:2,QTAU6,buy,4999.75,2,2026-06-16T14:38:05.000+00:00,2026-06-16 14:38:05.000 UTC,False,,False,,8,neither,0
sim:backtrader:backtest:3,QTBU6,sell,5010.25,3,2026-06-15T17:09:27.000+00:00,2026-06-15 17:09:27.000 UTC,True,5,False,,49,phantom,37.5
sim:backtrader:backtest:4,QTAU6,buy,4999.5,1,2026-06-16T17:32:15.000+00:00,2026-06-16 17:32:15.000 UTC,False,,True,0.371,1,real_only,6.25
sim:backtrader:backtest:5,QTAU6,sell,5002.75,2,2026-06-15T15:39:13.000+00:00,2026-06-15 15:39:13.000 UTC,True,5,False,,52,phantom,25
sim:backtrader:backtest:6,QTAU6,buy,5004,3,2026-06-15T19:04:05.000+00:00,2026-06-15 19:04:05.000 UTC,False,,False,,5,neither,0
sim:backtrader:backtest:7,QTBU6,buy,5008.75,1,2026-06-15T15:36:19.000+00:00,2026-06-15 15:36:19.000 UTC,True,5,False,,22,phantom,6.25


In [7]:
display(res.table()[["symbol", "side", "price", "entry_local", "backtest_filled", "real_filled", "outcome"]])

,symbol,side,price,entry_local,backtest_filled,real_filled,outcome
0,QTBU6,sell,5008.5,2026-06-16 14:38:42.000 UTC,True,True,both
1,QTAU6,buy,4999.5,2026-06-16 17:25:05.000 UTC,False,False,neither
2,QTBU6,buy,5008.25,2026-06-16 16:12:29.000 UTC,True,False,phantom
3,QTAU6,buy,5001.25,2026-06-15 15:05:05.000 UTC,False,True,real_only
4,QTAU6,buy,4999.75,2026-06-16 14:38:05.000 UTC,False,False,neither
5,QTBU6,sell,5010.25,2026-06-15 17:09:27.000 UTC,True,False,phantom
6,QTAU6,buy,4999.5,2026-06-16 17:32:15.000 UTC,False,True,real_only
7,QTAU6,sell,5002.75,2026-06-15 15:39:13.000 UTC,True,False,phantom
8,QTAU6,buy,5004,2026-06-15 19:04:05.000 UTC,False,False,neither
9,QTBU6,buy,5008.75,2026-06-15 15:36:19.000 UTC,True,False,phantom
